# Notebook 3: Risk Analysis (Medium - SQL + Python + Visualization)
Analyze loan risk using SQL for data retrieval and Python matplotlib for charts.

In [ ]:
%%sql -r risk_data
SELECT l.LOAN_ID, l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE,
       l.LOAN_STATUS, l.TERM_MONTHS, l.MONTHLY_PAYMENT,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.EMPLOYMENT_STATUS,
       a.DEBT_TO_INCOME_RATIO, a.REQUESTED_AMOUNT, a.APPROVED_AMOUNT
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
JOIN LOAN_DB.LENDING.LOAN_APPLICATIONS a ON l.CUSTOMER_ID = a.CUSTOMER_ID

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df = risk_data.copy()
df['IS_TROUBLED'] = df['LOAN_STATUS'].isin(['DELINQUENT', 'DEFAULT'])
print(f"Total loans: {len(df)}")
print(f"Troubled loans: {df['IS_TROUBLED'].sum()} ({df['IS_TROUBLED'].mean()*100:.1f}%)")
print(f"Troubled exposure: ${df.loc[df['IS_TROUBLED'], 'LOAN_AMOUNT'].sum():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df['LOAN_STATUS'].value_counts()
colors = {'CURRENT': '#2ecc71', 'DELINQUENT': '#f39c12', 'DEFAULT': '#e74c3c'}
bar_colors = [colors.get(s, '#95a5a6') for s in status_counts.index]
axes[0].bar(status_counts.index, status_counts.values, color=bar_colors)
axes[0].set_title('Loan Count by Status')
axes[0].set_ylabel('Count')

status_amounts = df.groupby('LOAN_STATUS')['LOAN_AMOUNT'].sum()
bar_colors2 = [colors.get(s, '#95a5a6') for s in status_amounts.index]
axes[1].bar(status_amounts.index, status_amounts.values, color=bar_colors2)
axes[1].set_title('Exposure by Status')
axes[1].set_ylabel('Amount ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for status in ['CURRENT', 'DELINQUENT', 'DEFAULT']:
    subset = df[df['LOAN_STATUS'] == status]
    ax.scatter(subset['CREDIT_SCORE'], subset['LOAN_AMOUNT'],
              label=status, alpha=0.7, s=80,
              color=colors.get(status, '#95a5a6'), edgecolors='white')

ax.set_xlabel('Credit Score')
ax.set_ylabel('Loan Amount ($)')
ax.set_title('Credit Score vs Loan Amount by Status')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

credit_bins = [0, 600, 660, 720, 780, 850]
credit_labels = ['Poor(<600)', 'Fair(600-659)', 'Good(660-719)', 'VGood(720-779)', 'Excellent(780+)']
df['CREDIT_TIER'] = pd.cut(df['CREDIT_SCORE'], bins=credit_bins, labels=credit_labels)

tier_risk = df.groupby('CREDIT_TIER', observed=True).agg(
    total=('LOAN_ID', 'count'),
    troubled=('IS_TROUBLED', 'sum')
)
tier_risk['trouble_rate'] = (tier_risk['troubled'] / tier_risk['total'] * 100)

axes[0].bar(tier_risk.index, tier_risk['trouble_rate'], color='#e74c3c', alpha=0.8)
axes[0].set_title('Trouble Rate by Credit Tier')
axes[0].set_ylabel('Trouble Rate (%)')
axes[0].tick_params(axis='x', rotation=30)

dti_bins = [0, 0.3, 0.4, 0.5, 1.0]
dti_labels = ['Low(<30%)', 'Moderate(30-40%)', 'High(40-50%)', 'Very High(50%+)']
df['DTI_TIER'] = pd.cut(df['DEBT_TO_INCOME_RATIO'], bins=dti_bins, labels=dti_labels)

dti_risk = df.groupby('DTI_TIER', observed=True).agg(
    total=('LOAN_ID', 'count'),
    troubled=('IS_TROUBLED', 'sum')
)
dti_risk['trouble_rate'] = (dti_risk['troubled'] / dti_risk['total'] * 100)

axes[1].bar(dti_risk.index, dti_risk['trouble_rate'], color='#f39c12', alpha=0.8)
axes[1].set_title('Trouble Rate by DTI Ratio')
axes[1].set_ylabel('Trouble Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
risk_summary = df.groupby('LOAN_TYPE').agg(
    loan_count=('LOAN_ID', 'count'),
    total_exposure=('LOAN_AMOUNT', 'sum'),
    troubled_count=('IS_TROUBLED', 'sum'),
    avg_credit_score=('CREDIT_SCORE', 'mean'),
    avg_dti=('DEBT_TO_INCOME_RATIO', 'mean'),
    avg_rate=('INTEREST_RATE', 'mean')
).round(2)
risk_summary['trouble_pct'] = (risk_summary['troubled_count'] / risk_summary['loan_count'] * 100).round(1)
risk_summary.sort_values('trouble_pct', ascending=False)